<a href="https://colab.research.google.com/github/rakshitha2006gowda-art/github-profile-analyzer/blob/main/Github_Profile_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# GitHub Profile Analyzer
# Python for AI Development

import requests


# 1. Get GitHub Profile
def get_github_profile(username):

    url = f"https://api.github.com/users/{username}"

    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            return response.json()

        elif response.status_code == 404:
            print("GitHub user not found.")
            return None

        elif response.status_code == 403:
            print("GitHub API rate limit reached.")
            return None

        else:
            print("Error:", response.status_code)
            return None

    except requests.exceptions.RequestException as error:
        print("Network error:", error)
        return None


# 2. Get Repositories
def get_repositories(username):

    url = f"https://api.github.com/users/{username}/repos"

    params = {
        "per_page": 100,
        "sort": "updated"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        if response.status_code == 200:
            return response.json()

        print("Unable to fetch repositories.")
        return []

    except requests.exceptions.RequestException as error:
        print("Network error:", error)
        return []


# 3. Classify Developer
def classify_profile(public_repos):

    if public_repos >= 20:
        return "Highly Active Developer"

    elif public_repos >= 10:
        return "Active Developer"

    elif public_repos >= 5:
        return "Growing Developer"

    else:
        return "Beginner Developer"


# 4. Analyze Repositories
def analyze_repositories(repositories):

    repo_names = []
    languages = []

    total_stars = 0
    total_forks = 0

    for repo in repositories:

        repo_names.append(
            repo.get("name", "Unknown")
        )

        language = repo.get("language")

        if language:
            languages.append(language)

        total_stars += repo.get(
            "stargazers_count", 0
        )

        total_forks += repo.get(
            "forks_count", 0
        )

    # Unique languages
    unique_languages = set(languages)

    # Count languages
    language_count = {}

    for language in languages:

        if language in language_count:
            language_count[language] += 1

        else:
            language_count[language] = 1

    # Most used language
    most_used_language = None

    if language_count:
        most_used_language = max(
            language_count,
            key=language_count.get
        )

    # Average stars
    average_stars = 0

    if repositories:
        average_stars = (
            total_stars / len(repositories)
        )

    return {
        "repo_names": repo_names,
        "total_repos": len(repositories),
        "total_stars": total_stars,
        "total_forks": total_forks,
        "average_stars": average_stars,
        "unique_languages": unique_languages,
        "language_count": language_count,
        "most_used_language": most_used_language
    }


# 5. Display Profile
def display_profile(profile):

    print("\n" + "=" * 50)
    print("              GITHUB PROFILE")
    print("=" * 50)

    print(
        "Name         :",
        profile.get("name") or "Not provided"
    )

    print(
        "Username     :",
        profile.get("login")
    )

    print(
        "Bio          :",
        profile.get("bio") or "Not provided"
    )

    print(
        "Location     :",
        profile.get("location") or "Not provided"
    )

    print(
        "Company      :",
        profile.get("company") or "Not provided"
    )

    print(
        "Public Repos :",
        profile.get("public_repos")
    )

    print(
        "Followers    :",
        profile.get("followers")
    )

    print(
        "Following    :",
        profile.get("following")
    )

    print(
        "Profile URL  :",
        profile.get("html_url")
    )


# 6. Display Repository Report
def display_repository_report(
    repositories,
    analysis
):

    print("\n" + "=" * 50)
    print("          REPOSITORY ANALYSIS")
    print("=" * 50)

    if not repositories:

        print("No public repositories found.")
        return

    # Display first 10 repositories
    for index, repo in enumerate(
        repositories[:10],
        start=1
    ):

        name = repo.get(
            "name",
            "Unknown"
        )

        language = repo.get(
            "language"
        ) or "Not specified"

        stars = repo.get(
            "stargazers_count",
            0
        )

        forks = repo.get(
            "forks_count",
            0
        )

        print(f"\n{index}. {name}")
        print("   Language :", language)
        print("   Stars    :", stars)
        print("   Forks    :", forks)

    print("\n" + "-" * 50)
    print("SUMMARY")
    print("-" * 50)

    print(
        "Repositories analyzed :",
        analysis["total_repos"]
    )

    print(
        "Total stars            :",
        analysis["total_stars"]
    )

    print(
        "Total forks            :",
        analysis["total_forks"]
    )

    print(
        "Average stars          :",
        round(
            analysis["average_stars"],
            2
        )
    )

    print(
        "Unique languages       :",
        analysis["unique_languages"]
    )

    print(
        "Most used language     :",
        analysis["most_used_language"]
        or "Not available"
    )


# 7. Find Most-Starred Repository
def find_most_starred_repository(
    repositories
):

    if not repositories:
        return None

    most_starred = repositories[0]

    for repo in repositories:

        if repo.get(
            "stargazers_count",
            0
        ) > most_starred.get(
            "stargazers_count",
            0
        ):

            most_starred = repo

    return most_starred


# 8. Search Repository
def search_repository(
    repositories,
    name
):

    for repo in repositories:

        if repo.get(
            "name",
            ""
        ).lower() == name.lower():

            return repo

    return None


# 9. Search Repositories by Language
def search_by_language(
    repositories,
    language
):

    results = []

    for repo in repositories:

        repo_language = repo.get(
            "language"
        )

        if (
            repo_language
            and repo_language.lower()
            == language.lower()
        ):

            results.append(
                repo.get("name")
            )

    return results


# 10. Main Function
def main():

    print("=" * 60)
    print("           GITHUB PROFILE ANALYZER")
    print("=" * 60)

    username = input(
        "Enter GitHub username: "
    ).strip()

    if not username:

        print("Username cannot be empty.")
        return

    print("\nFetching GitHub profile...")

    profile = get_github_profile(
        username
    )

    if profile is None:
        return

    print("Fetching repositories...")

    repositories = get_repositories(
        username
    )

    # Analyze
    analysis = analyze_repositories(
        repositories
    )

    # Developer classification
    level = classify_profile(
        profile.get(
            "public_repos",
            0
        )
    )

    # Display profile
    display_profile(profile)

    print(
        "\nDeveloper Level:",
        level
    )

    # Display repositories
    display_repository_report(
        repositories,
        analysis
    )

    # Most starred
    most_starred = (
        find_most_starred_repository(
            repositories
        )
    )

    if most_starred:

        print("\n" + "=" * 50)
        print("       MOST-STARRED REPOSITORY")
        print("=" * 50)

        print(
            "Repository :",
            most_starred.get("name")
        )

        print(
            "Stars      :",
            most_starred.get(
                "stargazers_count",
                0
            )
        )

        print(
            "Language   :",
            most_starred.get(
                "language"
            ) or "Not specified"
        )

    # Extra options
    print("\n" + "=" * 50)
    print("             EXTRA OPTIONS")
    print("=" * 50)

    print("1. Search Repository")
    print("2. Search by Language")
    print("3. Exit")

    choice = input(
        "\nEnter your choice: "
    ).strip()

    if choice == "1":

        name = input(
            "Enter repository name: "
        ).strip()

        repo = search_repository(
            repositories,
            name
        )

        if repo:

            print("\nRepository Found!")

            print(
                "Name     :",
                repo.get("name")
            )

            print(
                "Language :",
                repo.get("language")
                or "Not specified"
            )

            print(
                "Stars    :",
                repo.get(
                    "stargazers_count",
                    0
                )
            )

            print(
                "URL      :",
                repo.get("html_url")
            )

        else:

            print("\nRepository not found.")

    elif choice == "2":

        language = input(
            "Enter programming language: "
        ).strip()

        results = search_by_language(
            repositories,
            language
        )

        if results:

            print(
                f"\nRepositories using {language}:"
            )

            for repo in results:
                print("-", repo)

        else:

            print(
                f"\nNo repositories found using {language}."
            )

    elif choice == "3":

        print(
            "\nThank you for using the analyzer!"
        )

    else:

        print("\nInvalid choice.")

    print("\n" + "=" * 60)
    print("          ANALYSIS COMPLETED")
    print("=" * 60)


# Run the application
main()

           GITHUB PROFILE ANALYZER
Enter GitHub username: Rakshitha

Fetching GitHub profile...
Fetching repositories...

              GITHUB PROFILE
Name         : Rakshitha
Username     : Rakshitha
Bio          : Not provided
Location     : Banglore
Company      : Tresbu
Public Repos : 1
Followers    : 2
Following    : 0
Profile URL  : https://github.com/Rakshitha

Developer Level: Beginner Developer

          REPOSITORY ANALYSIS

1. game-of-life
   Language : XML
   Stars    : 0
   Forks    : 0

--------------------------------------------------
SUMMARY
--------------------------------------------------
Repositories analyzed : 1
Total stars            : 0
Total forks            : 0
Average stars          : 0.0
Unique languages       : {'XML'}
Most used language     : XML

       MOST-STARRED REPOSITORY
Repository : game-of-life
Stars      : 0
Language   : XML

             EXTRA OPTIONS
1. Search Repository
2. Search by Language
3. Exit

Enter your choice: 1
Enter repository name: